# Art Paint Pigment：NIR 高光譜影像 × 化學計量學（PLS 濃度預測）

以巴塞隆納大學 **Art Paint Pigment Concentrations** 資料集（IASIM-10 工作坊）示範完整流程：
讀取 DSO .mat → log(1/R) 前處理 → PCA 探索 → PLS 留一交叉驗證 → 逐像素預測產生濃度分布圖。

- 24 個油畫顏料樣品：普魯士藍（Prussian）、酞菁藍（Heliogen）、群青（Ultramarine）+ 油
- 影像 240×240 像素 × 207 波長（988.9–1674.7 nm），uint16 值 V = R × 65536
- 教學頁（互動圖表與測驗）：https://tai-shengyeh.github.io/spectraview/artpaint.html

> 資料使用限制：僅供教學；額外使用或發表須先取得 Prof. José F. García（jfgarcia@ub.edu）或 James Burger（james.burger@burgermetrics.com）同意。

## 0. 檔案路徑設定

**本機 Jupyter**：把 `DATA` 改成 ArtImageDataA 資料夾路徑。

**Google Colab**：先把 .mat 檔上傳到 Google Drive，執行下面的掛載 cell（會跳出授權），再把 `DATA` 改成 Drive 內的路徑。

In [ ]:
IN_COLAB = False
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/ArtImageDataA/'
else:
    DATA = './'  # 本機：ArtImageDataA 資料夾
print('IN_COLAB =', IN_COLAB, '| DATA =', DATA)

## 1. 讀取 DSO .mat 與還原影像

.mat 內是 PLS_Toolbox 的 DSO struct。注意 MATLAB 是 column-major，像素列還原成影像時要用 `order='F'`。

In [ ]:
import numpy as np, scipy.io as sio
import matplotlib.pyplot as plt

m    = sio.loadmat(DATA + 'PaintCubeB.mat')
dso  = m['PaintCubeB'][0, 0]
X    = dso['data'].astype(float) / 65536.0   # V = R*65536 -> 反射率 R, (57600, 207)
wl   = np.ravel(dso['axisscale'][1, 0])       # 207 個波長 (nm)
mask = m['PaintMask'].astype(int)             # (240, 240), 值 1-24
print(X.shape, wl[0], wl[-1], mask.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im0 = ax[0].imshow(X.mean(axis=1).reshape(240, 240, order='F'), cmap='viridis')
ax[0].set_title('mean reflectance'); plt.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(mask, cmap='turbo')
ax[1].set_title('PaintMask (sample id)'); plt.colorbar(im1, ax=ax[1])
plt.tight_layout()

## 2. 前處理與各樣品平均光譜

log(1/R) 擬吸收度讓訊號與濃度更接近線性（Beer–Lambert）。

⚠️ 前處理是超參數：本資料集若再加 SNV，LOO 的 RMSECV 會從 ~4.9% 惡化到 ~38.6%（樣品 #17 外插 + 整體吸收強度資訊被削掉）——前處理要用驗證結果來選。

In [ ]:
A = np.log10(1.0 / np.clip(X, 1e-6, None))
mask_f = mask.reshape(-1, order='F')          # 與 X 的列對齊
S = np.array([A[mask_f == k].mean(axis=0) for k in range(1, 25)])  # (24, 207)

for i, s in enumerate(S):
    c = '#123a63' if i < 8 else ('#0E7C7B' if i < 16 else '#5B4B8A')
    plt.plot(wl, s, color=c, lw=1)
plt.xlabel('wavelength (nm)'); plt.ylabel('log(1/R)')
plt.title('24 sample mean spectra (blue=Prussian, teal=Heliogen, purple=Ultramarine series)')

## 3. y 值與 PCA 探索

In [ ]:
ym   = sio.loadmat(DATA + 'ArtImageY.mat')
yTot = ym['yTotalPercent'][0, 0]['data'].astype(float)   # (57600, 4) 逐像素
y24  = np.array([yTot[mask_f == k].mean(axis=0) for k in range(1, 25)])
yP   = y24[:, 0]   # Prussian %
print('labels: Prussian, Heliogen, Ultramarine, Oil')
print(np.round(y24, 2))

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
scores = pca.fit_transform(S - S.mean(axis=0))
plt.scatter(scores[:, 0], scores[:, 1], c=yP, cmap='plasma')
for i in range(24):
    plt.annotate(i + 1, scores[i], fontsize=7)
plt.colorbar(label='Prussian %')
plt.xlabel('PC1 (%.1f%%)' % (100 * pca.explained_variance_ratio_[0]))
plt.ylabel('PC2 (%.1f%%)' % (100 * pca.explained_variance_ratio_[1]))

## 4. PLS 迴歸：留一交叉驗證（LOO-CV）

注意樣品 #17（唯一純群青）誤差最大：它被留出時訓練集中沒有相似組成，模型只能外插。

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import LeaveOneOut

for nlv in [2, 3, 4, 5, 6, 8]:
    pred = np.zeros(24)
    for tr, te in LeaveOneOut().split(S):
        pred[te] = PLSRegression(n_components=nlv).fit(S[tr], yP[tr]).predict(S[te]).ravel()
    print('LV=%d  RMSECV=%.2f %%' % (nlv, np.sqrt(np.mean((pred - yP) ** 2))))

In [ ]:
NLV = 5
pred = np.zeros(24)
for tr, te in LeaveOneOut().split(S):
    pred[te] = PLSRegression(n_components=NLV).fit(S[tr], yP[tr]).predict(S[te]).ravel()
rmsecv = np.sqrt(np.mean((pred - yP) ** 2))
r2 = 1 - np.sum((pred - yP) ** 2) / np.sum((yP - yP.mean()) ** 2)

plt.scatter(yP, pred, c=['#123a63'] * 8 + ['#0E7C7B'] * 8 + ['#5B4B8A'] * 8)
for i in range(24):
    plt.annotate(i + 1, (yP[i], pred[i]), fontsize=7)
lim = [yP.min() - 3, yP.max() + 3]
plt.plot(lim, lim, 'r--'); plt.xlabel('actual Prussian %'); plt.ylabel('LOO-CV predicted %')
plt.title('RMSECV=%.2f%%  R2(CV)=%.3f  (LV=%d)' % (rmsecv, r2, NLV))

## 5. 逐像素預測 → 濃度分布圖（chemical image）

用 Cube B 的像素訓練，預測**獨立拍攝**的 Cube C——這才是真正的外部驗證（隨機切像素會因相鄰像素高度相關而資料洩漏、高估表現）。

In [ ]:
mC = sio.loadmat(DATA + 'PaintCubeC.mat')
XC = np.log10(65536.0 / np.clip(mC['PaintCubeC'][0, 0]['data'].astype(float), 1, None))
maskC_f = mC['PaintMask'].astype(int).reshape(-1, order='F')

rng = np.random.default_rng(0)
idx = rng.choice(A.shape[0], 6000, replace=False)
pls_pix = PLSRegression(n_components=NLV).fit(A[idx], yTot[idx, 0])
predC = pls_pix.predict(XC).ravel()

plt.figure(figsize=(5.5, 4.5))
plt.imshow(predC.reshape(240, 240, order='F'), cmap='inferno', vmin=0, vmax=35)
plt.colorbar(label='predicted Prussian %'); plt.title('Cube C chemical image')

In [ ]:
# 檢核：Cube C 24 個樣品的平均預測 vs 已知濃度（RMSEP 應在 3-5% 左右）
predC_mean = np.array([predC[maskC_f == k].mean() for k in range(1, 25)])
rmsep = np.sqrt(np.mean((predC_mean - y24[:, 0]) ** 2))
print('cube C per-sample RMSEP = %.2f %%' % rmsep)

## 6. 練習題

1. 把 target 換成 **Heliogen**、**Ultramarine**，各自的最佳 LV 數與 RMSECV 是多少？
2. 對 `A` 加上 SNV 再跑一次第 4 節，觀察樣品 #17 的預測值怎麼變。
3. 用 `PaintDemo.mat` 內的 `PaintCubeA_ed`（80×240 裁切版）當訓練集重做第 5 節。
4. 把逐像素預測結果的直方圖畫出來，討論同一樣品內的空間變異來源。

完成後回教學頁做互動測驗：https://tai-shengyeh.github.io/spectraview/artpaint.html#sec-quiz